# 02 - Quality Control Contact Sheets

Generate per-slide thumbnail grids and a master contact sheet so you can visually
identify artifacts, debris, or missed tissue sections.

**Prerequisites:**
- Extracted tissue tiles from **notebook 01**
- Docker and the Conda environment already include the notebook dependencies
- For a local editable install, use `pip install -e ".[visualization]"`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from wsi_pipeline.qc_grid import run_qc_workflow

%load_ext autoreload
%autoreload 2

## Step 1: Configure Paths

In [ ]:
# Notebook 01 writes TIFF tiles here by default in Docker.
img_root_dir = Path("/output/tissue_sections").expanduser().resolve()
qc_output_dir = img_root_dir / "_qc_grids"

print(f"Image root:  {img_root_dir}")
print(f"QC output:   {qc_output_dir}")

## Step 2: Run the QC Workflow

`run_qc_workflow()` auto-consumes `tile_manifest.json` when notebook 01 has
written one, falls back to legacy filename parsing otherwise, renders per-slide
grids, writes a master contact sheet, and exports image statistics.

Parameters:
- **`thumb_size`** - thumbnail resolution in pixels
- **`columns`** - number of grid columns (or `"auto"`)
- **`label_mode`** - `"both"` shows per-slide and global indices
- **`backend`** - `"pil"` is deterministic; `"torch"` is an explicit opt-in

In [ ]:
qc_result = run_qc_workflow(
    input_dir=img_root_dir,
    output_dir=qc_output_dir,
    thumb_size=256,
    padding=0,
    columns="auto",
    label_mode="both",
    backend="pil",
    write_master=True,
    write_per_slide=True,
    write_stats=True,
)

if not qc_result.records:
    print(
        "No QC input found. Run notebook 01 first, or point img_root_dir at a "
        "legacy tile directory with renamed files."
    )
else:
    print(f"Loaded {len(qc_result.records)} QC record(s)")
    if qc_result.artifacts.records_manifest is not None:
        print(f"Manifest:  {qc_result.artifacts.records_manifest.name}")
    if qc_result.artifacts.master_contact_sheet is not None:
        print(f"Master:    {qc_result.artifacts.master_contact_sheet.name}")
    for p in qc_result.artifacts.per_slide_grids:
        print(f"Per-slide: {p.name}")
    if qc_result.artifacts.stats_csv is not None:
        print(f"Stats:     {qc_result.artifacts.stats_csv.name}")

## Step 3: Display the Master Contact Sheet

In [ ]:
master = qc_result.artifacts.master_contact_sheet
if master is not None and master.exists():
    plt.figure(figsize=(12, 12))
    plt.axis("off")
    plt.title("Master Contact Sheet")
    plt.imshow(Image.open(master))
    plt.show()
else:
    print("Master contact sheet not found.")

## Step 4: Display Per-Slide Grids

In [ ]:
per_slide = qc_result.artifacts.per_slide_grids
for grid_path in per_slide:
    plt.figure(figsize=(8, 6))
    plt.axis("off")
    plt.title(grid_path.stem)
    plt.imshow(Image.open(grid_path))
    plt.show()

## Step 5: Identify and Remove Artifacts

Look at the contact sheet above, note the label of any non-tissue tile (glass,
debris), and delete it from the output directory.  After deletion, re-run
`rename_outputs_by_overall_index()` in **notebook 01, step 5** to fix the
global index sequence and refresh `tile_manifest.json`.

In [ ]:
# Example: delete a problematic tile identified from the contact sheet
fname_to_delete = img_root_dir / "EXAMPLE_NAME.tif"
if fname_to_delete.exists():
    fname_to_delete.unlink()
    print(f"Deleted: {fname_to_delete.name}")
else:
    print(f"Not found (expected): {fname_to_delete.name}")

## Next Steps

- **Notebook 03** - Visualize extracted tissues in Neuroglancer
- **Notebook 04** - Prepare the image stack for EM-LDDMM registration

In [ ]:
# Re-run the same QC workflow call as Step 2 after fixing tiles in notebook 01
qc_result = run_qc_workflow(
    input_dir=img_root_dir,
    output_dir=qc_output_dir,
    thumb_size=256,
    padding=0,
    columns="auto",
    label_mode="both",
    backend="pil",
    write_master=True,
    write_per_slide=True,
    write_stats=True,
)
print(f"Regenerated {len(qc_result.records)} QC record(s)")

## Optional: Review Image Statistics

Review the per-image dimensions and intensity statistics written by
`run_qc_workflow()`.

In [ ]:
stats_csv = qc_result.artifacts.stats_csv
if stats_csv is None or not stats_csv.exists():
    print("No statistics CSV was written because no QC records were found.")
else:
    df = pd.read_csv(stats_csv)
    print(df.describe(include="all"))

    print(f"\nLoaded statistics from {stats_csv}")

## Next Steps

- **Notebook 03** - Visualize extracted tissues in Neuroglancer
- **Notebook 04** - Prepare the image stack for EM-LDDMM registration